In [ ]:
import numpy as np
import os
import torch
import torch.nn as nn
import torch.nn.functional as F
import trl
from contextlib import nullcontext
from datasets import load_dataset, Dataset
from packaging import version
from peft import get_peft_model, prepare_model_for_kbit_training, LoraConfig, AutoPeftModelForCausalLM
from transformers import Trainer, TrainingArguments, AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from trl import SFTTrainer, SFTConfig #, setup_chat_format, DataCollatorForCompletionOnlyLM

c:\Users\Shruti\anaconda3\envs\slm_env\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [ ]:
supported = torch.cuda.is_bf16_supported(including_emulation=False)
compute_dtype = (torch.bfloat16 if supported else torch.float32)

nf4_config = BitsAndBytesConfig(
   load_in_4bit=True,
   bnb_4bit_quant_type="nf4",
   bnb_4bit_use_double_quant=True,
   bnb_4bit_compute_dtype=compute_dtype
)

model_id = "microsoft/phi-3-mini-4k-instruct"
model_q4 = AutoModelForCausalLM.from_pretrained(model_id,
                                                device_map='cuda:0',
                                                dtype=compute_dtype,
                                                quantization_config=nf4_config)

model_q4 = prepare_model_for_kbit_training(model_q4)

config = LoraConfig(
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
    target_modules=[ "q_proj",
        "k_proj",
        "v_proj",
        "o_proj",
        "gate_proj",
        "up_proj",
        "down_proj",],
    #modules_to_save=['layernorm']
)
peft_model = get_peft_model(model_q4, config)

Loading weights: 100%|██████████| 195/195 [00:12<00:00, 15.18it/s]


In [ ]:
!nvidia-smi --query-gpu=index,utilization.gpu,memory.used,memory.total

index, utilization.gpu [%], memory.used [MiB], memory.total [MiB]
0, 4 %, 5902 MiB, 6141 MiB


In [ ]:
tokenizer = AutoTokenizer.from_pretrained(model_id)

dataset = load_dataset("dvgodoy/yoda_sentences", split="train")
dataset = dataset.rename_column("sentence", "prompt")
dataset = dataset.rename_column("translation_extra", "completion")

In [ ]:
tokenizer.pad_token, tokenizer.padding_side

('<|endoftext|>', 'left')

In [ ]:
dataset[0]

{'prompt': 'The birch canoe slid on the smooth planks.',
 'translation': 'On the smooth planks, the birch canoe slid.',
 'completion': 'On the smooth planks, the birch canoe slid. Yes, hrrrm.'}

In [ ]:
def make_messages_format_dataset(examples):
    """Converts prompt/completion pairs to conversational messages format."""
    if isinstance(examples["prompt"], list):
        output_texts = []
        for i in range(len(examples["prompt"])):
            converted_sample = [
                {"role": "user", "content": examples["prompt"][i]},
                {"role": "assistant", "content": examples["completion"][i]},
            ]
            output_texts.append(converted_sample)
        return {'messages': output_texts}
    else:
        converted_sample = [
            {"role": "user", "content": examples["prompt"]},
            {"role": "assistant", "content": examples["completion"]},
        ]
        return {'messages': converted_sample}

In [ ]:
dataset = dataset.map(make_messages_format_dataset, batched=True)
dataset[0]

{'prompt': 'The birch canoe slid on the smooth planks.',
 'translation': 'On the smooth planks, the birch canoe slid.',
 'completion': 'On the smooth planks, the birch canoe slid. Yes, hrrrm.',
 'messages': [{'content': 'The birch canoe slid on the smooth planks.',
   'role': 'user'},
  {'content': 'On the smooth planks, the birch canoe slid. Yes, hrrrm.',
   'role': 'assistant'}]}

In [ ]:
dataset = dataset.remove_columns(["prompt", "completion", "translation"])
dataset[0]

{'messages': [{'content': 'The birch canoe slid on the smooth planks.',
   'role': 'user'},
  {'content': 'On the smooth planks, the birch canoe slid. Yes, hrrrm.',
   'role': 'assistant'}]}

In [ ]:
def byofd_formatting_func(examples):
    output = []
    for i in range(len(examples["messages"])):
        output.append(
            tokenizer.apply_chat_template(
                conversation = examples["messages"][i], tokenize=False, padding = False, 
                truncation = False, 
                add_generation_prompt = False)
                )
    return {"text": output}

dataset = dataset.map(byofd_formatting_func, batched=True)
dataset[0]

{'messages': [{'content': 'The birch canoe slid on the smooth planks.',
   'role': 'user'},
  {'content': 'On the smooth planks, the birch canoe slid. Yes, hrrrm.',
   'role': 'assistant'}],
 'text': '<|user|>\nThe birch canoe slid on the smooth planks.<|end|>\n<|assistant|>\nOn the smooth planks, the birch canoe slid. Yes, hrrrm.<|end|>\n<|endoftext|>'}

In [ ]:
training_seq = dataset['text']
training_seq[0]

'<|user|>\nThe birch canoe slid on the smooth planks.<|end|>\n<|assistant|>\nOn the smooth planks, the birch canoe slid. Yes, hrrrm.<|end|>\n<|endoftext|>'

In [ ]:
len(training_seq)

720

In [ ]:
supported = torch.cuda.is_bf16_supported(including_emulation=False)
min_effective_batch_size = 8
lr = 3e-4
max_seq_length = 64
collator_fn = None
packing = (collator_fn is None)
steps = 20
num_train_epochs = 10

sft_config = SFTConfig(
    #output_dir='./future_name_on_the_hub',
    # Dataset
    packing=packing,
    packing_strategy='wrapped',
    max_length=max_seq_length,
    # Gradients / Memory
    gradient_checkpointing=True,
    gradient_checkpointing_kwargs={'use_reentrant': False},
    gradient_accumulation_steps=5,
    per_device_train_batch_size=min_effective_batch_size,
    auto_find_batch_size=True,
    # Training
    num_train_epochs=num_train_epochs,
    learning_rate=lr,
    # Env and Logging
    report_to='none',
    #logging_dir='./logs',
    logging_strategy='steps',
    logging_steps=steps,
    eval_strategy='steps',
    eval_steps=steps,
    save_strategy='steps',
    save_steps=steps,
    bf16=supported
)

In [ ]:
mvt_trainer = SFTTrainer(
    model=peft_model,
    processing_class=tokenizer,
    train_dataset=dataset.select(range(600)),
    eval_dataset=dataset.select(range(600, len(dataset))),
    args=sft_config
)

In [ ]:
!nvidia-smi --query-gpu=index,utilization.gpu,memory.used,memory.total

index, utilization.gpu [%], memory.used [MiB], memory.total [MiB]
0, 0 %, 3946 MiB, 6141 MiB


In [ ]:
mvt_trainer.train()

CheckpointError: torch.utils.checkpoint: A different number of tensors was saved during the original forward and recomputation.
Number of tensors saved during forward: 91
Number of tensors saved during recomputation: 34.

Tip: To see a more detailed error message, either pass `debug=True` to
`torch.utils.checkpoint.checkpoint(...)` or wrap the code block
with `with torch.utils.checkpoint.set_checkpoint_debug_enabled(True):` to
enable checkpoint‑debug mode globally.


In [ ]:
test = {"prompt": "There is bacon in this sandwich.",}
test_formatted = tokenizer.apply_chat_template(
                conversation = [
                    {"role": "user", "content": test["prompt"]},
                ], tokenize=False, padding = False, return_tensors="pt", add_generation_prompt = True)

print(test_formatted)
inputs = tokenizer(test_formatted, return_tensors="pt").to(peft_model.device)
with torch.no_grad():
    outputs = peft_model.generate(**inputs, max_new_tokens=64)
print(tokenizer.batch_decode(outputs, skip_special_tokens=True)[0])

`use_cache=True` is incompatible with gradient checkpointing. Setting `use_cache=False`.
Caching is incompatible with gradient checkpointing in Phi3DecoderLayer. Setting `past_key_values=None`.


<|user|>
There is bacon in this sandwich.<|end|>
<|assistant|>

There is bacon in this sandwich. In the first.


In [ ]:
with torch.no_grad():
    outputs = model_q4.generate(**inputs, max_new_tokens=64)
print(tokenizer.batch_decode(outputs, skip_special_tokens=True)[0])

There is bacon in this sandwich. In the first.
